# Train the image captioner on Kaggle

**Before running:**
1. Settings → Accelerator → **GPU** (T4 or P100). Turn **Internet on**.
2. Add data (right panel → *Add Input*): the **MS COCO 2014** images (`train2014/`, `val2014/`) and the **Karpathy split file** `dataset_coco.json`.
3. Update the two paths in the next cell to match where Kaggle mounted them (look under `/kaggle/input/`).

In [ ]:
COCO_IMAGES = "/kaggle/input/YOUR-COCO-DATASET"          # folder that contains train2014/ and val2014/
KARPATHY_JSON = "/kaggle/input/YOUR-KARPATHY-DATASET/dataset_coco.json"
!ls $COCO_IMAGES | head

In [ ]:
!git clone https://github.com/PiyushaKadam07/image-captioning-transformer.git
%cd image-captioning-transformer
!pip install -q -r requirements.txt

## 1. Prepare the splits

In [ ]:
!python scripts/prepare_data.py --karpathy $KARPATHY_JSON --out data
!ln -sfn $COCO_IMAGES data/images
!head -c 300 data/test.jsonl

## 2. Quick check (about 5 minutes)
Trains on 5,000 captions for one epoch to make sure everything works before the long runs.

In [ ]:
!python scripts/train.py --config configs/resnet50.yaml --subset 5000 --epochs 1 --output-dir runs/check

## 3. Full training
Each run takes several hours on a free GPU. Kaggle sessions stop after about 12 hours and allow limited GPU hours per week, so run one encoder per session. If a session ends, `last.pt` and `log.csv` in the output folder show how far it got.
To keep the checkpoints, **Save Version → Save & Run All** so `/kaggle/working` is kept as output.

In [ ]:
!python scripts/train.py --config configs/resnet50.yaml --output-dir /kaggle/working/runs/resnet50

In [ ]:
!python scripts/train.py --config configs/clip.yaml --output-dir /kaggle/working/runs/clip

## 4. Evaluate on the 5,000-image test split (beam search and greedy, for the ablation table)

In [ ]:
for enc in ["resnet50", "clip"]:
    ckpt = f"/kaggle/working/runs/{enc}/best.pt"
    !python scripts/evaluate.py --checkpoint {ckpt} --beam-size 3
    !python scripts/evaluate.py --checkpoint {ckpt} --beam-size 1
!python scripts/results_table.py /kaggle/working/runs

## 5. Try it on an image

In [ ]:
import glob
from IPython.display import Image as Show
sample = sorted(glob.glob(f"{COCO_IMAGES}/val2014/*.jpg"))[0]
!python scripts/caption.py --checkpoint /kaggle/working/runs/clip/best.pt --image {sample} --attention attention.png
Show("attention.png")